# 01 — API за 10 строк: изгиб круга

Мягкий шарнир, равномерная нагрузка; сверка с аналитикой расщепления.

In [ ]:
from plate_solver import Config, analytic
from plate_solver.geometry import make_circle
from plate_solver.plate import PlateBending

cfg = Config(q0=4.0, p=8, Q=256)
pb = PlateBending.from_config(make_circle(1.0), cfg)
_, cw = pb.solve_uniform(cfg.q0)
w0 = float(pb.deflection(cw, 0.0, 0.0))
w_ref = analytic.circular_plate_soft_hinge_wmax(cfg.q0, 1.0, cfg.D)
print(f"w(0) = {w0:.6e};  аналитика {w_ref:.6e};  rel = {abs(w0-w_ref)/w_ref:.2e}")

Поверхность прогиба (для сохранения фигуры используйте `viz.plot_deflection_surface(..., save=...)`).

In [ ]:
from plate_solver import viz

fig = viz.plot_deflection_surface(cfg, pb, cw)

## Квартет (D3): 3D-прогиб · карта σ · контактный планшет · сходимость
Через слой Result: fields.npz содержит всё для перерисовки (viz.replot).

In [ ]:

from plate_solver import viz
from plate_solver.dispatch import solve
from plate_solver.problem import Problem

res = solve(Problem.from_dict({
    "geometry": {"kind": "circle", "a": 1.0},
    "bc": {"type": "soft_hinge"},
    "load": {"type": "uniform", "q0": 4.0},
    "discretization": {"p": 8, "Q": 128, "grid_n": 48},
}))
Mx, My, Mxy = res.moments_on_grid()
viz.surface3d(res.Xg, res.Yg, res.w_grid, elev=30, azim=-50)
from plate_solver.ktn import stresses_faces

s = stresses_faces(Mx, My, Mxy, h=res.config.h, nu=res.config.nu, q_top=res.config.q0)
viz.stress_maps(res.Xg, res.Yg, s, components=["sx_top", "sx_bot"])